In [10]:
import polars as pl
from wjp_judicial_independence.config import PATH_DATA_RAW, PATH_DATA_INTERIM
from wjp_judicial_independence.sentiment import classify_sentiment
import polars.selectors as cs

# Filtrar noticias de independencia judicial de acuerdo a estrategia de modulo 1
MODULE1_STRATEGY = "embeddings" # "embeddings" | "llm" | "llm_api"

df_judicial_independence = pl.read_parquet(PATH_DATA_INTERIM/f"module1/df_{MODULE1_STRATEGY}_strategy_judicial_independence.parquet").filter(pl.col("is_judicial_independence"))
df_embeddings = pl.read_parquet(PATH_DATA_INTERIM/f"module1/df_embeddings_strategy_judicial_independence.parquet").select("country", "pillar", "event", cs.starts_with("score"))
df_judicial_independence = df_judicial_independence.join(df_embeddings, how="left", on = ["country", "pillar", "event"])

# Sentimientos: Amenaza vs Fortalecimiento de Independencia Judicial

Esto no es tan fácil como ver el impacto del evento. Pues puede que hayan dobles direcciones. Por ejemplo:

_"The Supreme Court ruled against the government's emergency decree"_

**Impacto:** Negativo. Malo para el gobierno, puede verse como disruptivo

**Sentimiento de Independencia Judicial:** Fortalecimiento. El tribunal actuó independientemente

Utilizaremos un LLM que se encargue de etiquetar si la noticia es amenaza, neutral o fortalecimiento en términos de independencia judicial.

In [11]:
SYSTEM_PROMPT = """You are a sentiment classifier for the World Justice Project (WJP). You will receive a news event that has already been identified as relevant to judicial independence. Your only task is to determine whether the event represents a threat to judicial independence or a strengthening of it.

## WHAT COUNTS AS A THREAT

The event describes an action, pattern, or situation that weakens the ability of courts and judges to operate independently. Examples:
- Politicized or non-transparent appointment or removal of judges.
- Executive or legislature ignoring, defying, or undermining court rulings.
- Political attacks, pressure campaigns, or retaliation against judges for their decisions.
- Laws that reduce judicial jurisdiction, pack courts, or curtail judicial review.
- Corruption, bribery, or influence trading involving judicial actors.
- Declining public trust or framing of courts as politically captured.
- Disciplinary proceedings used to punish judges for independent rulings.

## WHAT COUNTS AS A STRENGTHENING

The event describes an action, pattern, or situation that reinforces the ability of courts and judges to operate independently. Examples:
- Courts successfully blocking unlawful government actions or defending rights.
- Reinstatement of improperly suspended or dismissed judges.
- Anti-corruption investigations or accountability measures within the judiciary.
- Reforms that improve transparency in judicial appointments or tenure protections.
- Executive compliance with court rulings, especially on politically sensitive issues.
- Growing public confidence in judicial impartiality.
- International support, recognition, or conditionality reinforcing judicial autonomy.

## DECISION RULE

Ask: "Does this event make judicial independence stronger or weaker in practice?"
- Weaker → "threat"
- Stronger → "strengthening"
- Genuinely ambiguous or mixed → "neutral"

Respond with ONLY one word: threat, strengthening, or neutral. Nothing else."""

USER_MSG_TEMPLATE = (
    "Country: {country} | Pillar: {pillar}\n\n"
    "Event:\n{event}\n\nSentiment ('threat' or 'strengthening' or 'neutral'):"
)

In [12]:
LLM_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct" # modelos de la librería transformers, gratuitos
df_sentiment_with_llm = classify_sentiment(df_judicial_independence, "llm", model_name=LLM_MODEL_NAME, system_prompt=SYSTEM_PROMPT, user_msg_template=USER_MSG_TEMPLATE)
df_sentiment_with_llm.write_parquet(PATH_DATA_INTERIM/f"module2/sentiment/df_m1_{MODULE1_STRATEGY}_strategy_judicial_independence.parquet")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 245/245 [01:10<00:00,  3.49it/s]
